# Deep Agents — Revision Notebook
**Purpose:** concise, practical revision notes with Markdown explanations and separate Python code cells.

> **Mental model:** Deep Agents is an opinionated agent harness built on LangChain's agent abstractions and LangGraph runtime. It adds a ready-made set of capabilities for longer, multi-step work: planning, filesystem/context management, subagents, and summarization.

**Before running:** use a fresh virtual environment and install current compatible packages. Provider/model names depend on your account and provider integration. Some advanced APIs evolve; check the linked official docs if an example differs from your installed version.

## 1. Install and configure

The core package is `deepagents`. Add the provider integration for the model you intend to use (for example, `langchain-openai` or `langchain-anthropic`). Keep API keys in environment variables, not notebook source.

Useful references:
- [Deep Agents API reference](https://reference.langchain.com/python/deepagents/graph/create_deep_agent)
- [Deep Agents documentation](https://docs.langchain.com/oss/python/deepagents)

In [ ]:
%pip install -U deepagents
# Install only the provider integration you use, e.g.:
# %pip install -U langchain-openai
# %pip install -U langchain-anthropic

In [ ]:
import os

# Set these in your shell / notebook environment, not as hard-coded secrets.
# os.environ["OPENAI_API_KEY"] = "..."
# os.environ["ANTHROPIC_API_KEY"] = "..."

## 2. Where Deep Agents fits

| Layer | Responsibility |
|---|---|
| **LangChain** | Agent abstractions, model/tool integration, middleware |
| **LangGraph** | Graph execution, state, checkpoints, interrupts, streaming |
| **Deep Agents** | Opinionated setup for planning, filesystem tools, subagents, and context management |

Deep Agents is not a replacement for understanding LangGraph. It gives you a higher-level starting point; when you need custom execution flow or state behavior, LangGraph concepts still matter.

## 3. Create the smallest useful Deep Agent

`create_deep_agent(...)` returns a runnable agent. Call it with `.invoke(...)` similarly to other LangChain/LangGraph runnables.

The example below uses a provider/model string. Replace it with a model supported by your provider and installed integration.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="openai:YOUR_MODEL_NAME",
    system_prompt=(
        "You are a careful research assistant. "
        "Make a plan for multi-step tasks, verify claims, and summarize findings."
    ),
)

result = agent.invoke({
    "messages": [
        {"role": "user", "content": "Explain the difference between vector RAG and vectorless RAG."}
    ]
})

print(result["messages"][-1].content)

## 4. What comes built in?

The standard Deep Agents setup commonly provides:

- **Planning:** break a larger task into manageable steps.
- **Filesystem tools:** list, read, write, edit, search files (`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`).
- **Subagent delegation:** a `task` tool to delegate bounded work.
- **Summarization/context management:** reduce pressure on the model context as work grows.
- **Optional execution:** shell execution is available only when the selected backend supports the required sandbox execution protocol.

Important: the presence of a tool in the package does not mean every backend supports every capability. In particular, do not assume `execute` works with the default in-state backend.

## 5. Add your own tool

Use a plain Python callable for a small, stateless capability. The agent can decide when to call it; do not give tools unnecessary permissions.

In [ ]:
from deepagents import create_deep_agent

def word_count(text: str) -> int:
    """Return the number of whitespace-separated words."""
    return len(text.split())

agent_with_tool = create_deep_agent(
    model="openai:YOUR_MODEL_NAME",
    tools=[word_count],
    system_prompt="Use tools when useful. Be concise and accurate.",
)

result = agent_with_tool.invoke({
    "messages": [
        {"role": "user", "content": "Count the words in: 'Agents can plan and delegate work.'"}
    ]
})
print(result["messages"][-1].content)

## 6. Filesystem tools and the backend boundary

The filesystem tools let an agent keep intermediate notes or drafts outside the immediate conversation text. A backend determines **where** those files live and which operations are supported.

A common default is `StateBackend`: files are represented in agent state and can be checkpointed with a thread. It is useful for state-scoped work, but it is not the same as unrestricted access to your computer's filesystem.

**Rule:** treat filesystem access and shell execution as security boundaries. Use least privilege, isolate untrusted work, and avoid exposing secrets or production credentials.

In [ ]:
# Conceptual prompt to test the built-in filesystem tools:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": (
            "Create a short project brief in a file named brief.md, "
            "then read it back and summarize it."
        )}
    ]
})
print(result["messages"][-1].content)

## 7. Subagents: delegate bounded work

A subagent is a separate agent configured for a narrower task. The main agent can delegate through the `task` tool and receive a concise result.

Good delegation descriptions state:
1. the subagent's role,
2. the task boundary,
3. the expected output,
4. constraints or sources to use.

By default, a declarative subagent is isolated and receives the delegated task rather than the parent's full conversation. `mode="fork"` exists for continuing with inherited context, but is experimental in the current reference.

In [ ]:
from deepagents import create_deep_agent

subagents = [
    {
        "name": "researcher",
        "description": "Research one topic and return a concise evidence-based summary.",
        "system_prompt": (
            "Focus on the assigned topic. Separate confirmed facts from uncertainty. "
            "Return key findings and open questions."
        ),
    },
    {
        "name": "reviewer",
        "description": "Review a draft for correctness, gaps, and unclear reasoning.",
        "system_prompt": (
            "Critique the supplied draft. Identify unsupported claims, missing caveats, "
            "and concrete improvements."
        ),
    },
]

delegating_agent = create_deep_agent(
    model="openai:YOUR_MODEL_NAME",
    subagents=subagents,
    system_prompt="Delegate focused work when it helps; synthesize the results yourself.",
)

result = delegating_agent.invoke({
    "messages": [
        {"role": "user", "content": "Prepare a concise comparison of RAG and long-context prompting."}
    ]
})
print(result["messages"][-1].content)

## 8. Planning is not the same as guaranteed execution

A plan helps structure work; it does not prove that every step was completed correctly. For important tasks:
- ask for verifiable intermediate artifacts,
- inspect outputs,
- validate calculations with deterministic code,
- define stopping criteria,
- evaluate final answers against requirements.

For reproducible workflows, make the requested deliverable and acceptance criteria explicit in the prompt.

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": (
            "Task: draft a one-page project proposal. "
            "Plan first, create a draft file, check it against the requirements "
            "(goal, scope, risks, milestones), revise any missing sections, "
            "then report the file name and remaining uncertainties."
        )}
    ]
})
print(result["messages"][-1].content)

## 9. Memory vs filesystem vs conversation state

| Mechanism | Best mental model | Typical purpose |
|---|---|---|
| Conversation messages | What was said in this run/thread | Immediate interaction history |
| Graph state/checkpoint | Runtime state saved by LangGraph | Resume a thread, inspect/update state |
| Agent filesystem | Named working artifacts | Drafts, notes, large intermediate results |
| Deep Agents `memory` / `AGENTS.md` | Persistent instructions/context loaded for the agent | Stable project conventions or guidance |

Do not assume that a file, checkpoint, or memory instruction is globally persistent across all users or all sessions. Persistence depends on backend, checkpointer, storage, and configuration.

## 10. Reusing a thread / checkpointing

For durable thread state, configure a checkpointer appropriate to your environment and pass a stable `thread_id` in `config`. The in-memory saver is useful for demonstrations, not durable production storage.

Exact imports and persistence setup can vary by installed LangGraph version; verify against the current LangGraph persistence docs.

In [ ]:
# Illustrative pattern; persistence backend and imports may vary by version.
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

persistent_agent = create_deep_agent(
    model="openai:YOUR_MODEL_NAME",
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "demo-thread-001"}}

first = persistent_agent.invoke(
    {"messages": [{"role": "user", "content": "My project is a document QA assistant."}]},
    config=config,
)

second = persistent_agent.invoke(
    {"messages": [{"role": "user", "content": "What is my project?"}]},
    config=config,
)

print(second["messages"][-1].content)

## 11. Human approval and guardrails

Deep Agents can be combined with LangChain middleware and human-in-the-loop patterns. Use approval gates for high-impact actions (external writes, payments, destructive operations, sending messages, production changes).

A reliable pattern:
1. validate inputs before the agent acts,
2. keep tools narrow and permissioned,
3. pause for approval before consequential tool calls,
4. validate outputs after execution,
5. log tool calls and decisions without logging secrets.

HITL configuration and interrupt behavior depend on the tool configuration and a compatible checkpointer. Review the current `create_deep_agent` API before copying an exact `interrupt_on` configuration into production.

In [ ]:
# Design sketch: define which tools need approval in your deployment.
# Check the installed Deep Agents / LangChain version for the current
# `interrupt_on` schema and required checkpointer configuration.
#
# agent = create_deep_agent(
#     model="openai:YOUR_MODEL_NAME",
#     tools=[read_only_tool, consequential_write_tool],
#     interrupt_on={"consequential_write_tool": True},
#     checkpointer=checkpointer,
# )

## 12. Streaming and async execution

Because the result is a LangGraph-backed runnable, common runnable patterns such as streaming and async invocation are available, subject to the chosen model and integrations. Use streaming when you want incremental progress or to build a live UI.

`stream_mode` options and emitted event shapes are version-sensitive; inspect the installed LangGraph docs for the precise mode you need.

In [ ]:
# Basic streaming pattern (event shape depends on stream_mode and version)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Make a three-step plan to learn agent evaluation."}]},
    stream_mode="updates",
):
    print(chunk)

In [ ]:
# Async usage, e.g. inside an async function:
# result = await agent.ainvoke({
#     "messages": [{"role": "user", "content": "Summarize this task."}]
# })

## 13. Skills and project instructions

Skills package reusable task instructions and, optionally, supporting resources. Deep Agents can load skill sources from configured paths. Project instruction files such as `AGENTS.md` can provide stable conventions or domain guidance.

Keep instructions:
- specific enough to guide decisions,
- small enough to remain useful,
- version-controlled when appropriate,
- free of credentials and sensitive data.

Paths are interpreted relative to the configured backend/root; verify that the files are actually available to the agent.

In [ ]:
# Illustrative configuration; make sure this path exists in the selected backend.
skill_aware_agent = create_deep_agent(
    model="openai:YOUR_MODEL_NAME",
    skills=["/skills/project-writing/"],
    memory=["/memory/AGENTS.md"],
    system_prompt="Follow project instructions and state any assumptions.",
)

## 14. When to use Deep Agents vs a basic agent vs custom LangGraph

| Choose | When |
|---|---|
| Basic LangChain agent | A straightforward tool loop is enough |
| **Deep Agents** | Multi-step tasks benefit from planning, working files, context management, or delegation |
| Custom LangGraph workflow | You need explicit deterministic routing, tightly controlled state transitions, specialized retries, or a domain-specific graph |

Deep Agents is a useful default for open-ended work, not a requirement for every agent application.

## 15. Common failure modes

- **Tool overload:** too many tools with overlapping descriptions confuse tool selection.
- **Vague subagents:** delegation produces generic results when scope/output are unclear.
- **Context bloat:** large file reads or tool results crowd out the task; summarize and retrieve selectively.
- **False completion:** the agent says it finished without checking artifacts.
- **Unsafe execution:** file/shell tools have broader access than the task requires.
- **Unclear persistence:** developers assume state or files persist beyond the configured backend/checkpointer.
- **Model mismatch:** the selected model does not reliably support tool calling or the required context length.
- **Version drift:** copied code uses a parameter that changed between package versions.

## 16. Production checklist

- [ ] Explicitly choose and pin compatible package versions.
- [ ] Use a model tested for reliable tool calling.
- [ ] Set a clear system prompt and concrete acceptance criteria.
- [ ] Give tools least-privilege access.
- [ ] Use a sandbox for execution; do not assume default state storage supports shell commands.
- [ ] Gate high-impact actions with human approval.
- [ ] Configure a durable checkpointer/store where required.
- [ ] Log traces, failures, latency, token usage, and tool outcomes safely.
- [ ] Evaluate task success, factuality, tool correctness, cost, and latency.
- [ ] Test prompt injection in user input, files, and tool results.
- [ ] Verify generated files and outputs before reporting success.

## 17. Quick revision sheet

- **Entry point:** `create_deep_agent(...)`
- **Run:** `.invoke({"messages": [...]})`
- **Extra capabilities:** planning, filesystem/context management, summarization, delegation.
- **Custom capability:** pass Python callables / LangChain tools through `tools=[...]`.
- **Subagents:** define focused specs with `name`, `description`, and optional prompt/tools/model.
- **State persistence:** LangGraph checkpointer + stable `thread_id`; backend controls file storage behavior.
- **Execution safety:** `execute` requires a sandbox-capable backend.
- **Best use:** open-ended, multi-step work with intermediate artifacts.
- **Not magic:** plans and final answers still require verification and evaluation.

## 18. Self-test

Try answering these without looking above:

1. What does Deep Agents add beyond a basic tool-calling agent?
2. What is the difference between LangGraph state and filesystem artifacts?
3. Why might `execute` fail with the default backend?
4. When should you use a subagent rather than another tool call?
5. What must be configured for resumable thread state?
6. What kinds of actions should require human approval?
7. When is a custom LangGraph workflow preferable to Deep Agents?

### Official references
- [Create a Deep Agent — API reference](https://reference.langchain.com/python/deepagents/graph/create_deep_agent)
- [Deep Agents overview / package reference](https://reference.langchain.com/python/deepagents/deepagents)
- [Subagents reference](https://reference.langchain.com/python/deepagents/middleware/subagents)
- [Filesystem middleware reference](https://reference.langchain.com/python/deepagents/middleware/filesystem)
- [Deep Agents documentation](https://docs.langchain.com/oss/python/deepagents)